In [ ]:
# In[1]:

import json
from datetime import datetime
import pytz

# Load the task.json file
task_directory = 'rca100/cases/t048'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window', {})
alert_start = alert_window.get('start')
alert_end = alert_window.get('end')
alert_entity = task_data.get('alert_entity')

# Convert alert time window to UTC+8 timezone
timezone = pytz.timezone('Asia/Shanghai')
alert_start_utc8 = datetime.fromisoformat(alert_start).astimezone(timezone) if alert_start else None
alert_end_utc8 = datetime.fromisoformat(alert_end).astimezone(timezone) if alert_end else None

alert_title, alert_start_utc8, alert_end_utc8, alert_entity

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred between 19:52:22 and 20:01:32 on April 11, 2026 (UTC+8). The alert was triggered by the operation `product-catalog::oteldemo.ProductCatalogService/GetProduct` in the APM domain.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', datetime.datetime(2026, 4, 11, 19, 52, 22, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), datetime.datetime(2026, 4, 11, 20, 1, 32, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), {'entity_id': '1948ddf615871ac6fc18d01f72817e86', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/GetProduct', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Extract unique KPIs and their corresponding components
metrics_data['metric'] = metrics_data['metric'].astype(str)  # Ensure metric column is string
metrics_data['entity_name'] = metrics_data['entity_name'].astype(str)  # Ensure entity_name column is string
available_kpis = metrics_data[['metric', 'entity_name']].drop_duplicates()

available_kpis

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs (e.g., `node_ready_status`, `node_memory_usage_rate`, `node_cpu_usage_rate`, `error_rate`) associated with different components, including nodes and services like `frontend` and `frontend-proxy`. There are 299 unique combinations of KPIs and components available for analysis.

The original code execution output of IPython Kernel is also provided below for reference:

metric                      entity_name
0           node_ready_status                                 
2145   node_memory_usage_rate                                 
3289    node_not_ready_status                                 
5720     node_disk_usage_rate                                 
8294      node_cpu_usage_rate                                 
...                       ...                              ...
89472              error_rate  frontend-proxy-67d64dc4f8-8c99b
89476              error_rate        frontend-6df7d557f5-nbvwh
89478              error_rate        frontend-6df7d557f5-nz7g4
89480              error_rate        frontend-6df7d557f5-sgfcb
89482              error_rate  frontend-proxy-67d64dc4f8-xz2fd

[299 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter metrics data for the `product-catalog` service and its operation `GetProduct`
product_catalog_metrics = metrics_data[
    metrics_data['entity_name'].str.contains('product-catalog', case=False)
]

# Ensure 'value' column is numeric for calculations
product_catalog_metrics['value'] = pd.to_numeric(product_catalog_metrics['value'], errors='coerce')

# Calculate global thresholds (e.g., P95) for each KPI of the `product-catalog` service
global_thresholds = product_catalog_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
global_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

global_thresholds

```
Out[3]:
```
The global P95 thresholds for the `product-catalog` service KPIs are as follows:
- `avg_request_latency_seconds`: 0.0000537 seconds
- `cpu`: 0.0271
- `latency`: 0.00004899 seconds
- `mem`: 42,485,350 bytes
- `request_count`: 12,323
- `workload`: 19,322

These thresholds can be used to identify potential anomalies in the service's performance.

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0  avg_request_latency_seconds   5.369880e-05
1                          cpu   2.714267e-02
2                      latency   4.898551e-05
3                          mem   4.248535e+07
4                request_count   1.232330e+04
5                     workload   1.932200e+04```
```

In [ ]:
# In[4]:

# Convert the alert time window to microseconds for filtering
alert_start_microseconds = int(alert_start_utc8.timestamp() * 1_000_000)
alert_end_microseconds = int(alert_end_utc8.timestamp() * 1_000_000)

# Filter metrics data for the alert time window and `product-catalog` service KPIs
filtered_metrics = product_catalog_metrics[
    (product_catalog_metrics['time'] >= alert_start_microseconds) &
    (product_catalog_metrics['time'] <= alert_end_microseconds)
]

# Merge filtered metrics with global thresholds to identify anomalies
anomalies = filtered_metrics.merge(global_thresholds, on='metric')
anomalies['is_anomalous'] = anomalies['value'] > anomalies['p95_threshold']

# Filter only anomalous rows
anomalous_kpis = anomalies[anomalies['is_anomalous']]

anomalous_kpis

```
Out[4]:
```
During the alert time window, several KPIs for the `product-catalog` service exceeded their global P95 thresholds, indicating anomalies. These include:
- `avg_request_latency_seconds` and `latency`, with values slightly above their thresholds.
- `request_count` and `workload`, showing higher-than-expected values.
- `cpu` and `mem`, with usage exceeding their thresholds.

These anomalies suggest potential performance issues or abnormal traffic patterns affecting the `product-catalog` service during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set                         entity_id                                        entity_name  ...         value                            metric_set_id service p95_threshold  is_anomalous
28   1775908612000000    apm       apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  5.380096e-05  apm@metric_set@apm.metric.apm.operation    None  5.369880e-05          True
29   1775908642000000    apm       apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  5.416860e-05  apm@metric_set@apm.metric.apm.operation    None  5.369880e-05          True
40   1775908402000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.261000e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
43   1775908492000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.255300e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
44   1775908522000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.233100e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
50   1775908702000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.315700e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
51   1775908732000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.306600e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
52   1775908762000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.304300e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
53   1775908792000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.276800e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
54   1775908822000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.344800e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
55   1775908852000000    apm       apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  1.276100e+04  apm@metric_set@apm.metric.apm.operation    None  1.232330e+04          True
107  1775908702000000    apm        apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...  1.241000e+04   apm@metric_set@apm.metric.apm.instance    None  1.232330e+04          True
111  1775908822000000    apm        apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...  1.276800e+04   apm@metric_set@apm.metric.apm.instance    None  1.232330e+04          True
142  1775908612000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.401245e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
143  1775908642000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.651583e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
144  1775908672000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.475757e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
145  1775908702000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.470892e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
148  1775908792000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.464289e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
150  1775908852000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.429402e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
151  1775908882000000    apm        apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...  5.510007e-05   apm@metric_set@apm.metric.apm.instance    None  5.369880e-05          True
156  1775908372000000    apm  apm.service.legacy                                                                      product-catalog  ...  2.714865e-02                                     None    None  2.714267e-02          True
172  1775908492000000    apm  apm.service.legacy                                                                      product-catalog  ...  2.716674e-02                                     None    None  2.714267e-02          True
176  1775908522000000    apm  apm.service.legacy                                                                      product-catalog  ...  2.716288e-02                                     None    None  2.714267e-02          True
189  1775908612000000    apm  apm.service.legacy                                                                      product-catalog  ...  4.944997e-05                                     None    None  4.898551e-05          True
193  1775908642000000    apm  apm.service.legacy                                                                      product-catalog  ...  4.997350e-05                                     None    None  4.898551e-05          True
203  1775908702000000    apm  apm.service.legacy                                                                      product-catalog  ...  1.990800e+04                                     None    None  1.932200e+04          True
207  1775908732000000    apm  apm.service.legacy                                                                      product-catalog  ...  1.975900e+04                                     None    None  1.932200e+04          True
211  1775908762000000    apm  apm.service.legacy                                                                      product-catalog  ...  1.967600e+04                                     None    None  1.932200e+04          True
215  1775908792000000    apm  apm.service.legacy                                                                      product-catalog  ...  1.932500e+04                                     None    None  1.932200e+04          True
218  1775908822000000    apm  apm.service.legacy                                                                      product-catalog  ...  4.248781e+07                                     None    None  4.248535e+07          True
219  1775908822000000    apm  apm.service.legacy                                                                      product-catalog  ...  2.042300e+04                                     None    None  1.932200e+04          True
224  1775908882000000    apm  apm.service.legacy                                                                      product-catalog  ...  2.822254e-02                                     None    None  2.714267e-02          True

[32 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure relevant columns are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces for the `product-catalog` service during the alert time window
product_catalog_traces = traces_data[
    (traces_data['serviceName'].str.contains('product-catalog', case=False)) &
    (traces_data['startTime'] >= alert_start_microseconds * 1_000) &  # Convert to nanoseconds
    (traces_data['endTime'] <= alert_end_microseconds * 1_000)        # Convert to nanoseconds
]

# Identify faulty traces by checking for non-zero status codes or error messages
product_catalog_traces['statusCode'] = pd.to_numeric(product_catalog_traces['statusCode'], errors='coerce')
faulty_traces = product_catalog_traces[
    (product_catalog_traces['statusCode'] != 0) | 
    (product_catalog_traces['statusMessage'].notna())
]

# Group by serviceName to identify the most downstream faulty service
faulty_services = faulty_traces.groupby('serviceName').size().reset_index(name='fault_count')
faulty_services_sorted = faulty_services.sort_values(by='fault_count', ascending=False)

faulty_services_sorted

```
Out[5]:
```
The trace analysis shows that the `product-catalog` service is the most downstream faulty service in the call chain, with 9,828 faulty traces during the alert time window. This indicates that the issue originates within the `product-catalog` service itself.

The original code execution output of IPython Kernel is also provided below for reference:

serviceName  fault_count
0  product-catalog         9828```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the `product-catalog` service during the alert time window
product_catalog_logs = logs_data[
    (logs_data['_container_name_'].str.contains('product-catalog', case=False, na=False)) &
    (logs_data['_time_'] >= alert_start_microseconds) &
    (logs_data['_time_'] <= alert_end_microseconds)
]

# Filter logs for error messages or patterns indicating root cause
error_logs = product_catalog_logs[product_catalog_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[6]:
```
The log analysis for the `product-catalog` service during the alert time window did not reveal any error messages or patterns indicating the root cause of the anomalies. The logs contain no entries with error-related content.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns]```
```